# Adem coboundaries: the even prime

This notebook follows the paper [*A cochain level proof of Adem relations in the mod 2 Steenrod algebra*](../papers/AdemRelations.pdf) and uses `comch` to construct its Adem coboundaries in the Barratt-Eccles and surjection operads.

## Introduction

For mod-$2$ cohomology, the Steenrod squares satisfy the Adem relations

$$
Sq^a Sq^b = \sum_i \binom{b-1-i}{a-2i} Sq^{a+b-i} Sq^i,
\qquad a < 2b.
$$

The paper proves these relations at the cochain level. Instead of only showing that a certain sum of Steenrod operations vanishes in cohomology, it constructs explicit cochains whose coboundaries are the relevant Adem sums.

In `comch`, the pair of integers $(q,p)$ with $q > p \geq 0$ indexes the universal classes used in the paper; these indices are not, in general, the exponents of one composite $Sq^qSq^p$. The complete Barratt-Eccles coboundary consists of a homotopy component and two non-symmetric corrections. After table reduction to the surjection operad, it can be evaluated on simplicial chains to give a cochain representative of an Adem coboundary for any homogeneous mod-$2$ cocycle $\alpha$.

## The Barratt-Eccles Adem homotopy

Let

$$
\widetilde{x}_i = (e,(12),e,\ldots,(12)^i) \in \mathcal E(2).
$$

The Adem construction uses two homotopies. The first is the homotopy $J_\Psi$ between the inclusion of $EV_4$ and the map $\Psi = EZ \circ \Phi \circ AW$ built from the Alexander-Whitney and Eilenberg-Zilber maps. The second is the homotopy $J_{(23)}$ associated to the inner automorphism by the transposition $(23) \in \Sigma_4$.

The method `BarrattEccles.adem_homotopy(q, p)` returns the normalized Barratt-Eccles element

$$
J_\Psi(\widetilde{x}_q \times \widetilde{x}_p)
+ J_\Psi(\widetilde{x}_p \times \widetilde{x}_q)
+ J_{(23)}(\widetilde{x}_q \times \widetilde{x}_p)
\in \mathcal E(4).
$$

This is the homotopy component, not yet the complete cochain witness. The latter also contains the two non-symmetric terms $\widehat{NS}_{q,p}$ and $\widehat{NS}_{p,q}$ from [BMM]. The method `BarrattEccles.adem_coboundary(q, p)` returns their sum with the homotopy:

$$
A_{q,p}=\widetilde A_{q,p}+\widehat{NS}_{q,p}+\widehat{NS}_{p,q}.
$$

It is this complete element whose action gives the cochain appearing on the right-hand side of the Adem identity.

In [ ]:
from comch import BarrattEccles

def preview(element, max_terms=8):
    """Return a readable preview of a possibly long chain."""
    if len(element) <= max_terms:
        return str(element)
    head = dict(list(element.items())[:max_terms])
    shown = element.create(head)
    return f'{shown} + ... ({len(element) - max_terms} more terms)'

In [ ]:
for q, p in [(1, 0), (2, 1)]:
    coboundary = BarrattEccles.adem_coboundary(q, p)
    print(f'A_{{{q},{p}}} in E(4): degree {coboundary.degree}, {len(coboundary)} terms')
    print(preview(coboundary))
    print()

## Table reduction

The table reduction morphism $TR \colon \mathcal E \to \mathcal X$ sends Barratt-Eccles elements to surjection elements. Since the Adem elements here are defined over $\mathbb F_2$, the Berger-Fresse and McClure-Smith conventions have the same coefficients.

The low-level method `Surjection.adem_homotopy(q, p)` is the table reduction of the homotopy component. For cochain computations, use `Surjection.adem_coboundary(q, p)`, the table reduction of the complete element including both corrections.

In [ ]:
from comch import Surjection


for q, p in [(1, 0), (2, 1)]:
    surjection = Surjection.adem_coboundary(q, p)
    print(f'TR(A_{{{q},{p}}}) in X(4): degree {surjection.degree}, {len(surjection)} terms')
    print(preview(surjection))
    print()

## Action on simplicial chains

Simplicial cochains are algebras over the surjection operad. For a mod-$2$ cocycle $\alpha$, the Adem coboundary associated to $(q,p)$ is determined by the action of the surjection element on chains:

$$
A_{q,p}(\alpha)(c) =
(\alpha \otimes \alpha \otimes \alpha \otimes \alpha)\, a_{q,p}(c).
$$

The method `Steenrod.adem_coboundaries(q, p, n)` evaluates this chain map on the top $n$-simplex. Since $\alpha$ is homogeneous, the output is filtered to summands where all four tensor factors have the same dimension.

For cubical chains, a direct action of the Barratt-Eccles operad is under construction; until it lands, `shape='cube'` raises `NotImplementedError`.

In [ ]:
from comch import Steenrod


examples = [
    (1, 0, 3, 'simplex'),
    (2, 1, 4, 'simplex'),
]

for q, p, n, shape in examples:
    answer = Steenrod.adem_coboundaries(q, p, n, shape=shape)
    print(f'A_{{{q},{p}}} on the top {n}-{shape}: {len(answer)} terms')
    print(preview(answer))
    print()

try:
    Steenrod.adem_coboundaries(1, 0, 1, shape='cube')
except NotImplementedError as message:
    print(f'shape=cube: NotImplementedError: {message}')

A_{2,1} on the top 4-simplex: 22 terms
((0,1,3),(1,2,3),(0,3,4),(0,1,3)) + ((0,1,3),(1,2,3),(0,1,4),(1,3,4)) + ((0,2,4),(2,3,4),(0,1,4),(1,2,4)) + ((0,2,4),(2,3,4),(0,2,4),(0,1,2)) + ((0,1,4),(0,1,4),(1,2,3),(1,3,4)) + ((0,1,4),(0,1,4),(2,3,4),(1,2,4)) + ((0,1,4),(0,1,4),(1,3,4),(1,2,3)) + ((0,1,4),(0,1,4),(1,2,4),(2,3,4)) + ... (14 more terms)

shape=cube: NotImplementedError: the action of the Barratt-Eccles operad on cubical chains is not yet implemented


## An explicit Adem relation at the cochain level

We now verify, on an explicit cocycle, the full cochain-level identity proved in [BMM]. For parameters $(q, p)$ and a cocycle $\alpha$ of degree $n$ it reads

$$
\sum_\ell \binom{q-\ell}{q-2\ell} Sq^{2n-p-\ell} Sq^{n-q+\ell}(\alpha)
\; + \;
\sum_\ell \binom{p-\ell}{p-2\ell} Sq^{2n-q-\ell} Sq^{n-p+\ell}(\alpha)
\; = \;
\delta A_{q,p}(\alpha),
$$

where every Steenrod square is the cochain $Sq^k(\beta) = \beta \smallsmile_{|\beta|-k} \beta$, not just its cohomology class.

The method `Surjection.adem_coboundary(q, p)` returns the complete element $A_{q,p}$. Its two ingredients play different roles:

- $\eta_{q,p}(\alpha)$ is the evaluation of the Adem homotopy constructed in this notebook. It is the genuinely new datum: the chain-level witness that the two transpositions-conjugate cycles in $B\Sigma_4$ agree up to boundary.

- $NS(\alpha)$ is a classical correction made of cup products of Steenrod squares. It comes from the non-symmetric part $\partial \widehat{NS}_{q,p}$ of $\bar\Phi(x_q \otimes x_p)$ in [BMM]: since that part is a *boundary* in the operad, its evaluation on a tensor of cocycles is a coboundary and is invisible in cohomology. It is deliberately absent from the low-level `adem_homotopy`, but included by `adem_coboundary`. Explicitly, $NS$ is the action on $\alpha^{\otimes 4}$ of the surjection element

$$
\sum_{\substack{0 < a \leq \ell \\ a \equiv \ell \;\mathrm{mod}\; \mathbb Z}}
\binom{p-\ell-a}{p-2\ell} \binom{p-\ell+a}{p-2\ell}\,
\theta_{q-p+2\ell+1} \circ \big( \theta_{p-\ell-a},\, \theta_{p-\ell+a} \big)
\;\; + \;\; (p \leftrightarrow q),
$$

where $\theta_i = (1,2,1,\ldots) \in \mathcal X(2)$ is the cup-$i$ surjection and $\ell, a$ run over half-integers. Each summand acts as $Sq^{n-p+\ell+a}(\alpha) \smallsmile_{q-p+2\ell+1} Sq^{n-p+\ell-a}(\alpha)$.

We take $(q,p) = (3,1)$ and $n = 2$. The binomial coefficients leave a single term on the left-hand side: the relation becomes

$$
Sq^1 Sq^1(\alpha) = \delta A_{3,1}(\alpha).
$$

This is the cochain-level incarnation of the simplest Adem relation, $Sq^1 Sq^1 = 0$: the left-hand side is a nonzero cochain, and the right-hand side exhibits it explicitly as a coboundary.

In [ ]:
from itertools import combinations

from comch import SimplicialElement, SurjectionElement


def theta(i):
    """The cup-i surjection element (1,2,1,2,...) of degree i."""
    key = tuple(1 if j % 2 == 0 else 2 for j in range(i + 2))
    return SurjectionElement({key: 1}, torsion=2,
                             convention='Berger-Fresse')


homotopy_31 = Surjection.adem_homotopy(3, 1)
coboundary_31 = Surjection.adem_coboundary(3, 1)
# Over F_2, subtraction and addition agree.
correction_31 = coboundary_31 + homotopy_31

print('A_{3,1} consists of:')
print(f'  homotopy component: {len(homotopy_31)} terms')
print(f'  NS corrections:     {len(correction_31)} terms')
print(f'  complete element:    {len(coboundary_31)} terms')

We represent a mod-$2$ cochain by the set of basis simplices on which it evaluates to $1$. Evaluating the action of a surjection element on a tensor of cochains is then a matter of pairing each summand of the action with the sets:

$$
s(\alpha_1 \otimes \cdots \otimes \alpha_r)(c) =
(\alpha_1 \otimes \cdots \otimes \alpha_r)\big(s(c)\big).
$$

Our cocycle lives on the standard $5$-simplex: we take $\alpha = \delta u$ for the $1$-cochain $u$ dual to $\{03, 04, 12, 14, 15, 34\}$. Being a coboundary, $\alpha$ is automatically a cocycle of degree $2$; on a simplex every cocycle of positive degree is of this form.

In [ ]:
def evaluate(surjection, cochains, simplex):
    """Pair the action of a surjection element on a simplex with cochains."""
    chain = surjection(SimplicialElement({(simplex,): 1}, torsion=2))
    return sum(v for k, v in chain.items()
               if all(face in cochain
                      for face, cochain in zip(k, cochains))) % 2


cells = {d: list(combinations(range(6), d + 1)) for d in range(6)}


def coboundary_of_dual(support, dim):
    """The mod-2 coboundary of the cochain dual to the given simplices."""
    answer = set()
    for c in cells[dim + 1]:
        count = sum(1 for t in range(len(c))
                    if c[:t] + c[t + 1:] in support)
        if count % 2:
            answer.add(c)
    return answer


u = {(0, 3), (0, 4), (1, 2), (1, 4), (1, 5), (3, 4)}
alpha = coboundary_of_dual(u, 1)
print(f'alpha = delta(u) is supported on {len(alpha)} triangles:')
print(sorted(alpha))

First the left-hand side. At the cochain level $Sq^1(\alpha) = \alpha \smallsmile_1 \alpha$ has degree $3$, and applying $Sq^1$ again means cupping the result with itself along $\smallsmile_2$. We locate a $4$-simplex on which the composition does not vanish, so that the verification below cannot be mistaken for an identity between zeros.

In [ ]:
sq1_alpha = {c for c in cells[3]
             if evaluate(theta(1), (alpha, alpha), c)}
sq1sq1_alpha = {c for c in cells[4]
                if evaluate(theta(2), (sq1_alpha, sq1_alpha), c)}

print(f'Sq^1(alpha) is supported on {len(sq1_alpha)} tetrahedra')
print(f'Sq^1 Sq^1(alpha) is supported on {len(sq1sq1_alpha)} 4-simplices:')
print(sorted(sq1sq1_alpha))

witness = (0, 1, 3, 4, 5)
print(f'\nvalue of Sq^1 Sq^1(alpha) on {witness}:',
      int(witness in sq1sq1_alpha))

Now the right-hand side. A coboundary evaluated on a simplex is the original cochain evaluated on its boundary, so we evaluate the complete element returned by `adem_coboundary` over the five facets of the witness simplex. We also split it into its homotopy and correction components to make their separate contributions visible.

In [ ]:
facets = [witness[:t] + witness[t + 1:] for t in range(len(witness))]
delta_eta = sum(evaluate(homotopy_31, (alpha,) * 4, facet)
                for facet in facets) % 2
delta_ns = sum(evaluate(correction_31, (alpha,) * 4, facet)
               for facet in facets) % 2
delta_full = sum(evaluate(coboundary_31, (alpha,) * 4, facet)
                 for facet in facets) % 2

print('delta(eta(alpha))  on the witness:', delta_eta)
print('delta(NS(alpha))   on the witness:', delta_ns)
print('delta(A_{3,1})      on the witness:', delta_full)
print('Sq^1 Sq^1(alpha)   on the witness:', int(witness in sq1sq1_alpha))
print('\nAdem relation verified at the cochain level:',
      delta_full == int(witness in sq1sq1_alpha))

### When the correction becomes visible

In the example above $\delta(NS(\alpha))$ happened to vanish, so the homotopy alone carried the relation. To see the correction at work, take $(q, p) = (2, 1)$, still with cocycles of degree $2$. Now the left-hand side vanishes identically at the cochain level: $Sq^2 Sq^1$ occurs once in each of the two sums and cancels mod $2$, while $Sq^3 Sq^0(\alpha) = Sq^3(\alpha) = 0$ because $3$ exceeds the degree of $\alpha$. The identity therefore asserts

$$
\delta\big(\eta_{2,1}(\alpha)\big) = \delta\big(NS(\alpha)\big),
$$

and for the cocycle below both sides equal $1$ on the top simplex. The two cochains $\eta_{2,1}(\alpha)$ and $NS(\alpha)$ are individually *not* cocycles: their coboundaries agree and are nonzero, while their sum `Surjection.adem_coboundary(2, 1)` has zero coboundary on this cell. This is the precise sense in which the cup-product correction, though invisible in cohomology, is genuinely present at the cochain level.

In [ ]:
eta_21 = Surjection.adem_homotopy(2, 1)
coboundary_21 = Surjection.adem_coboundary(2, 1)
correction_21 = coboundary_21 + eta_21

u_2 = {(0, 1), (0, 2), (0, 3), (2, 3), (2, 4), (4, 5)}
alpha_2 = coboundary_of_dual(u_2, 1)

top = tuple(range(6))
facets = [top[:t] + top[t + 1:] for t in range(len(top))]
delta_eta_21 = sum(evaluate(eta_21, (alpha_2,) * 4, facet)
                   for facet in facets) % 2
delta_ns_21 = sum(evaluate(correction_21, (alpha_2,) * 4, facet)
                  for facet in facets) % 2
delta_full_21 = sum(evaluate(coboundary_21, (alpha_2,) * 4, facet)
                    for facet in facets) % 2

print('delta(eta(alpha))  on the top 5-simplex:', delta_eta_21)
print('delta(NS(alpha))   on the top 5-simplex:', delta_ns_21)
print('delta(A_{2,1})     on the top 5-simplex:', delta_full_21)
print('\nBoth nonzero and equal, as the relation with vanishing'
      '\nleft-hand side requires:', delta_eta_21 == delta_ns_21 == 1)

## References

[BMM] G. Brumfiel, A. Medina-Mardones, and J. Morgan. "A cochain level proof of Adem relations in the mod 2 Steenrod algebra."

[BF] C. Berger and B. Fresse. "Combinatorial operad actions on cochains." *Mathematical Proceedings of the Cambridge Philosophical Society* 137 (2004), no. 1, 135-174. [DOI](https://doi.org/10.1017/S0305004103007138)